# Attention Compression

For our first example, we will show how we can use compressors to accelerate the matrix multiplication in an attention mechanism. In transformers, the attention step is simply a sequence of matrix multiplications. 
First, we will load the required packages the first being `RandLinearAlgebra` and the second being `LinearAlgebra`.

In [12]:
using RandLinearAlgebra
using LinearAlgebra

Once we have loaded these packages, we will begin by defining a function that does the attention multiplications. Because Julia makes changing the precision of variables easy we will be sure to write it with flexible types.

In [13]:
function attention(X, Wq, Wk, Wv)
    Q = X * Wq
    K = X * Wk
    V = X * Wv
    d_k = size(K, 2)
    # make sure that hte scaling takes into account the type of the elements in the matrix
    scores = Q * K' / sqrt(eltype(X)(d_k))
    # softmax over each row
    scores = scores .- maximum(scores, dims=2)
    weights = exp.(scores) ./ sum(exp.(scores), dims=2)
    return weights * V
end

attention (generic function with 1 method)

Now we will incorporate compressors into the attention mechanism. This will require the use of the `complete_compressor` function and one of the many possible `Compressor` data types.

In [14]:
function compressed_systems(Wq, Wk, Wv; compression_dim)
    # here we specify a Gaussian Compressor
    specs = Gaussian(cardinality=Right(), compression_dim=compression_dim, type=eltype(Wq))
    # Here we form the Gaussian Compressor Recipe for each matrix
    compressors = [complete_compressor(specs, transpose(W)) for W in [Wq, Wk, Wv]]
    # Here we perform the compression of each inputted matrix
    return [W'*comp for (comp, W) in zip(compressors, [Wq, Wk, Wv])]
end

compressed_systems (generic function with 1 method)

Finally, we will now do some comparisons. On the CPU with `Float32`.

In [16]:
# Simulated Data
seq_len, d_model, d_k = 2^12, 2^10, 2^10
compression_dim = 32 

X_cpu   = randn(Float32, seq_len, d_model)
Wq_cpu  = randn(Float32, d_model, d_k)
Wk_cpu  = randn(Float32, d_model, d_k)
Wv_cpu  = randn(Float32, d_model, d_k)
Weights_cpu = (Wq_cpu, Wk_cpu, Wv_cpu)

# compressor the systems
ctime = @elapsed Weights_cpu_c = compressed_systems(Weights_cpu..., compression_dim=compression_dim)
println("=== CPU ===")
t1 = @elapsed A1 = attention(X_cpu, Weights_cpu...); println("CPU: $(t1)")
t2 = @elapsed AC = attention(X_cpu, Weights_cpu_c...); println("CPU-C: $(t2)"); println("CPU Compression:$(ctime)")

=== CPU ===
CPU: 0.48226675
CPU-C: 0.22243
CPU Compression:0.006988625
